# Gemma-2-9B QLoRA — SUBMISSION (inference)
Loads base Gemma-2-9B + the trained LoRA adapter, predicts the test set with **A/B-swap test-time augmentation** to cancel position bias, and writes `submission.csv`. Enable **GPU**, set **Internet = OFF**. Attach: the base **Gemma-2-9B** model, the **adapter dataset** (from training output), and the **competition** data.

In [ ]:
# Internet is OFF here, so we can't pip-install. Neutralize huggingface_hub @strict
# (defensively) so the image's transformers can import Gemma2Config offline.
try:
    import huggingface_hub, huggingface_hub.dataclasses as _hfd
    _noop = lambda cls=None, **kw: (cls if cls is not None else (lambda c: c))
    _hfd.strict = _noop; huggingface_hub.strict = _noop
    print('patched huggingface_hub.strict')
except Exception as e:
    print('no strict patch needed:', e)

In [ ]:
import torch, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel
MAX_LEN = 512    # keep the 25K x2-TTA scoring re-run well under the 9h cap
BATCH = 16

In [ ]:

import json, re, os, glob, numpy as np, pandas as pd

TARGETS = ["winner_model_a", "winner_model_b", "winner_tie"]

def find_dir(pattern):
    hits = glob.glob(pattern, recursive=True)
    assert hits, f"nothing matched {pattern} under /kaggle/input"
    return os.path.dirname(hits[0])

def parse_list(x):
    if isinstance(x, list): return [str(t) for t in x]
    if not isinstance(x, str): return [""]
    try: v = json.loads(x)
    except Exception: return [x]
    if isinstance(v, list): return ["" if t is None else str(t) for t in v]
    return ["" if v is None else str(v)]

def join(x): return "\n".join(parse_list(x))

def build_text(prompt, resp_a, resp_b, max_chars=7000):
    """Structured prompt for the classifier. Truncate each field head+tail."""
    def clip(s, n):
        s = s or ""
        return s if len(s) <= n else s[: n // 2] + " ... " + s[-n // 2 :]
    return (
        "You are judging which chatbot response a human prefers.\n\n"
        "### Prompt\n" + clip(prompt, max_chars // 3) +
        "\n\n### Response A\n" + clip(resp_a, max_chars // 3) +
        "\n\n### Response B\n" + clip(resp_b, max_chars // 3) +
        "\n\n### Which is preferred? A, B, or tie."
    )


In [ ]:
BASE = find_dir('/kaggle/input/**/config.json')
ADAPTER = find_dir('/kaggle/input/**/adapter_config.json')
COMP = find_dir('/kaggle/input/**/test.csv')
print('BASE =', BASE, '\nADAPTER =', ADAPTER, '\nCOMP =', COMP)

In [ ]:
# Load in fp16 sharded across all GPUs (T4 x2 -> ~29GB) instead of 4-bit, so we don't
# depend on a bitsandbytes version we can't pip-install offline. A 4-bit-trained LoRA
# adapter applies fine to an fp16 base for inference.
tok = AutoTokenizer.from_pretrained(ADAPTER)
if tok.pad_token is None: tok.pad_token = tok.eos_token
base = AutoModelForSequenceClassification.from_pretrained(
    BASE, num_labels=3, dtype=torch.float16, device_map='auto')
base.config.pad_token_id = tok.pad_token_id
# The image's torchao is too old and peft's check RAISES instead of skipping. Our adapter
# is plain LoRA (no torchao), so force the check to False before loading the adapter.
import peft.import_utils, peft.tuners.lora.torchao as _pt
peft.import_utils.is_torchao_available = _pt.is_torchao_available = lambda: False
model = PeftModel.from_pretrained(base, ADAPTER).eval()
print('loaded; device map:', getattr(model, 'hf_device_map', 'single'))

In [ ]:
test = pd.read_csv(f'{COMP}/test.csv')
P = [join(x) for x in test['prompt']]
A = [join(x) for x in test['response_a']]
B = [join(x) for x in test['response_b']]

@torch.no_grad()
def predict(texts):
    # Sort by length so each batch pads to its own longest row (not a global max),
    # cutting wasted compute; scatter results back to original order.
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    probs = np.zeros((len(texts), 3), dtype=np.float32)
    for s in range(0, len(order), BATCH):
        idx = order[s:s+BATCH]
        enc = tok([texts[i] for i in idx], truncation=True, max_length=MAX_LEN,
                  padding=True, return_tensors='pt').to(model.device)
        p = torch.softmax(model(**enc).logits.float(), dim=1).cpu().numpy()
        for j, i in enumerate(idx):
            probs[i] = p[j]
    return probs

In [ ]:
# Forward view (A,B) and swapped view (B,A); swapping cancels position bias.
fwd = predict([build_text(p, a, b) for p, a, b in zip(P, A, B)])
swp = predict([build_text(p, b, a) for p, a, b in zip(P, A, B)])
# In the swapped view, class a<->b flip, tie stays.
swp = swp[:, [1, 0, 2]]
prob = (fwd + swp) / 2

In [ ]:
sub = test[['id']].copy()
sub[TARGETS] = prob
sub.to_csv('/kaggle/working/submission.csv', index=False)
print(sub.shape); sub.head()